# Part G: Model Comparison & Evaluation
**Robust Regression Engine**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

In [ ]:
df = pd.read_csv(r"/mnt/user-data/uploads/Advanced_Regression_HousePrice_Dataset_3800_-_Advanced_Regression_HousePrice_Dataset_3800_csv.csv")

TARGET    = 'house_price_inr'
DROP_COLS = ['property_id', 'sale_date']
FEATURES  = [col for col in df.columns if col not in [TARGET] + DROP_COLS]

X = df[FEATURES]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# SVR subset
np.random.seed(42)
idx   = np.random.choice(len(X_train_scaled), 800, replace=False)
X_svr = X_train_scaled[idx]
y_svr = y_train.values[idx]

print("Data Ready")

## Task 22 — Train All Models & Evaluate

In [ ]:
# ── Ridge ──────────────────────────────────────────────────
ridge = Ridge(alpha=1000)
ridge.fit(X_train_scaled, y_train)
y_pred_ridge = ridge.predict(X_test_scaled)

print("===== Ridge Regression (L2) =====")
print("Training Score :", ridge.score(X_train_scaled, y_train))
print("Testing Score  :", ridge.score(X_test_scaled, y_test))
print("MSE  :", mean_squared_error(y_test, y_pred_ridge))
print("RMSE :", np.sqrt(mean_squared_error(y_test, y_pred_ridge)))
print("MAE  :", mean_absolute_error(y_test, y_pred_ridge))

In [ ]:
# ── Lasso ──────────────────────────────────────────────────
lasso = Lasso(alpha=1000, max_iter=50000)
lasso.fit(X_train_scaled, y_train)
y_pred_lasso = lasso.predict(X_test_scaled)

print("===== Lasso Regression (L1) =====")
print("Training Score :", lasso.score(X_train_scaled, y_train))
print("Testing Score  :", lasso.score(X_test_scaled, y_test))
print("MSE  :", mean_squared_error(y_test, y_pred_lasso))
print("RMSE :", np.sqrt(mean_squared_error(y_test, y_pred_lasso)))
print("MAE  :", mean_absolute_error(y_test, y_pred_lasso))

In [ ]:
# ── Decision Tree ──────────────────────────────────────────
dt = DecisionTreeRegressor(max_depth=8, random_state=42)
dt.fit(X_train, y_train)
y_pred_dt = dt.predict(X_test)

print("===== Decision Tree =====")
print("Training Score :", dt.score(X_train, y_train))
print("Testing Score  :", dt.score(X_test, y_test))
print("MSE  :", mean_squared_error(y_test, y_pred_dt))
print("RMSE :", np.sqrt(mean_squared_error(y_test, y_pred_dt)))
print("MAE  :", mean_absolute_error(y_test, y_pred_dt))

In [ ]:
# ── Random Forest ──────────────────────────────────────────
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

print("===== Random Forest =====")
print("Training Score :", rf.score(X_train, y_train))
print("Testing Score  :", rf.score(X_test, y_test))
print("MSE  :", mean_squared_error(y_test, y_pred_rf))
print("RMSE :", np.sqrt(mean_squared_error(y_test, y_pred_rf)))
print("MAE  :", mean_absolute_error(y_test, y_pred_rf))

In [ ]:
# ── SVR ────────────────────────────────────────────────────
svr = SVR(kernel='rbf', C=1e7, epsilon=1e5)
svr.fit(X_svr, y_svr)
y_pred_svr = svr.predict(X_test_scaled)

print("===== SVR (RBF) =====")
print("Testing Score :", r2_score(y_test, y_pred_svr))
print("MSE  :", mean_squared_error(y_test, y_pred_svr))
print("RMSE :", np.sqrt(mean_squared_error(y_test, y_pred_svr)))
print("MAE  :", mean_absolute_error(y_test, y_pred_svr))

## Task 23 — Compare All Models

In [ ]:
models_result = {
    'Ridge (L2)':    {'train': ridge.score(X_train_scaled, y_train), 'test': ridge.score(X_test_scaled, y_test),   'rmse': np.sqrt(mean_squared_error(y_test, y_pred_ridge)), 'mae': mean_absolute_error(y_test, y_pred_ridge)},
    'Lasso (L1)':    {'train': lasso.score(X_train_scaled, y_train), 'test': lasso.score(X_test_scaled, y_test),   'rmse': np.sqrt(mean_squared_error(y_test, y_pred_lasso)), 'mae': mean_absolute_error(y_test, y_pred_lasso)},
    'Decision Tree': {'train': dt.score(X_train, y_train),           'test': dt.score(X_test, y_test),             'rmse': np.sqrt(mean_squared_error(y_test, y_pred_dt)),    'mae': mean_absolute_error(y_test, y_pred_dt)},
    'Random Forest': {'train': rf.score(X_train, y_train),           'test': rf.score(X_test, y_test),             'rmse': np.sqrt(mean_squared_error(y_test, y_pred_rf)),    'mae': mean_absolute_error(y_test, y_pred_rf)},
    'SVR (RBF)':     {'train': svr.score(X_svr, y_svr),              'test': r2_score(y_test, y_pred_svr),         'rmse': np.sqrt(mean_squared_error(y_test, y_pred_svr)),   'mae': mean_absolute_error(y_test, y_pred_svr)},
}

df_results = pd.DataFrame(models_result).T
df_results.columns = ['Train R2', 'Test R2', 'RMSE', 'MAE']
print(df_results.round(4).to_string())

In [ ]:
# Test Score Bar Chart
comparison = pd.DataFrame({
    'Model': list(models_result.keys()),
    'Test Score': [v['test'] for v in models_result.values()]
})

comparison.plot(
    x='Model',
    y='Test Score',
    kind='bar',
    figsize=(10, 5),
    legend=False
)

plt.title("Model Test Score Comparison (R2)")
plt.ylabel("R2 Score")
plt.xticks(rotation=20)
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# RMSE Comparison
comparison_rmse = pd.DataFrame({
    'Model': list(models_result.keys()),
    'RMSE': [v['rmse'] for v in models_result.values()]
})

comparison_rmse.plot(
    x='Model',
    y='RMSE',
    kind='bar',
    figsize=(10, 5),
    legend=False,
    color='tomato'
)

plt.title("Model RMSE Comparison")
plt.ylabel("RMSE (INR)")
plt.xticks(rotation=20)
plt.grid(True)
plt.tight_layout()
plt.show()

## Task 24 — Identify Overfitting / Underfitting

In [ ]:
print(f"{'Model':<22} {'Train R2':>10} {'Test R2':>10} {'Gap':>8}  Verdict")
print("-" * 68)

for name, row in df_results.iterrows():
    gap = row['Train R2'] - row['Test R2']
    if gap > 0.15:
        verdict = "OVERFITTING"
    elif row['Test R2'] < 0.5:
        verdict = "UNDERFITTING"
    else:
        verdict = "Good Fit"
    print(f"{name:<22} {row['Train R2']:>10.4f} {row['Test R2']:>10.4f} {gap:>8.4f}  {verdict}")